In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
import torchvision
from tqdm import tqdm

%matplotlib inline

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('running on', device)

SAVE_FIGURES = False  # set to True to write the figures used on the slides to ./figures
def savefig(name):
    if SAVE_FIGURES:
        os.makedirs('figures', exist_ok=True)
        plt.savefig(f'figures/{name}.png', dpi=150, bbox_inches='tight')

In [ ]:
%%sh
# Download the data - you need to do this only once
wget --no-verbose --output-document=image_dog.jpg https://github.com/chrirupp/cv_course/raw/main/data/image_dog.jpg
wget --no-verbose --output-document=image_me.jpg https://github.com/chrirupp/cv_course/raw/main/data/image_me.jpg
wget --no-verbose --output-document=image_cc_1.jpg https://github.com/chrirupp/cv_course/raw/main/data/image_cc_1.jpg
wget --no-verbose --output-document=image.jpg https://github.com/chrirupp/cv_course/raw/main/data/image.jpg

# Lecture 8 - Transformers

Attention from scratch, how a Vision Transformer turns an image into tokens, what its learned positional embeddings look like, how far its heads look, and which axes the normalisation layers normalise over.

Run this notebook on Google Colab (see the [readme](https://github.com/chrirupp/cv_course)). Cells that load large pretrained models are faster with a GPU runtime (*Edit > Notebook settings > T4 GPU*).

## Attention as a soft lookup

A dictionary lookup returns the value whose key matches the query exactly. Attention returns a *weighted average* of all values, weighted by how well each key matches the query: $\text{softmax}(\mathbf{q}^\top \mathbf{k}_i / \tau)$. Here, in 1D: the keys are positions $x_i$, the values are noisy measurements $y_i$ of a function, and the similarity is $-(q - x_i)^2$. The temperature $\tau$ controls how many keys contribute (cf. the softmax temperature in Lecture 6).

In [ ]:
rng = np.random.default_rng(0)
keys = np.sort(rng.uniform(0, 10, 40))
values = np.sin(keys) + rng.normal(0, 0.25, len(keys))
queries = np.linspace(0, 10, 400)

def soft_lookup(queries, keys, values, tau):
    scores = -(queries[:, None] - keys[None, :]) ** 2 / tau        # (queries, keys)
    weights = np.exp(scores - scores.max(1, keepdims=True))
    weights /= weights.sum(1, keepdims=True)                        # softmax over the keys
    return weights @ values, weights

fig, axs = plt.subplots(2, 3, figsize=(20, 8), gridspec_kw={'height_ratios': [2, 1]})
for col, tau in enumerate([0.01, 0.3, 5.0]):
    out, w = soft_lookup(queries, keys, values, tau)
    axs[0, col].scatter(keys, values, c='k', s=15, label='(key, value) pairs')
    axs[0, col].plot(queries, np.sin(queries), 'g--', alpha=0.5, label='true function')
    axs[0, col].plot(queries, out, 'r-', label='attention output')
    axs[0, col].set_title(f'temperature {tau}')
    axs[0, col].legend(loc='lower left', fontsize=8)
    q = 200
    axs[1, col].bar(keys, w[q], width=0.15)
    axs[1, col].axvline(queries[q], color='r')
    axs[1, col].set_title(f'attention weights of the query q={queries[q]:.1f}')
savefig('transformer_soft_lookup')
plt.show()

In [ ]:
# scaled dot-product attention with several heads, written out, and checked against PyTorch
def attention(Q, K, V):
    # Q: (heads, n_queries, d), K, V: (heads, n_keys, d)
    A = torch.softmax(Q @ K.transpose(-1, -2) / np.sqrt(Q.shape[-1]), dim=-1)   # (heads, n_queries, n_keys)
    return A @ V, A

heads, n, d = 4, 6, 16
Q, K, V = torch.randn(3, heads, n, d).unbind(0)
out, A = attention(Q, K, V)
print('same as F.scaled_dot_product_attention:', torch.allclose(out, F.scaled_dot_product_attention(Q, K, V), atol=1e-6))
print('attention weights of head 0 (rows sum to one):\n', A[0].numpy().round(2))

## From an image to tokens

The Vision Transformer (ViT-B/16) cuts a 224x224 image into a 14x14 grid of 16x16 patches, flattens each patch to a vector of $16 \cdot 16 \cdot 3 = 768$ numbers, and projects it linearly (a convolution with kernel size and stride 16). A learned class token is prepended and a learned positional embedding added: the transformer then sees a set of 197 tokens.

In [ ]:
# weakly supervised pre-training on 3.6B Instagram images, then a linear classifier on ImageNet (224x224 inputs)
weights = torchvision.models.ViT_B_16_Weights.IMAGENET1K_SWAG_LINEAR_V1
vit = torchvision.models.vit_b_16(weights=weights).to(device).eval()
preprocess = weights.transforms()

def load(name):
    image = cv2.cvtColor(cv2.imread(name), cv2.COLOR_BGR2RGB)
    s = min(image.shape[:2])
    y0, x0 = (image.shape[0] - s) // 2, (image.shape[1] - s) // 2
    return cv2.resize(image[y0:y0+s, x0:x0+s], (224, 224), interpolation=cv2.INTER_AREA)

dog = load('image_dog.jpg')
patches = dog.reshape(14, 16, 14, 16, 3).transpose(0, 2, 1, 3, 4)     # (14, 14, 16, 16, 3)
fig = plt.figure(figsize=(16, 10))
gs = fig.add_gridspec(2, 2, height_ratios=[4, 1])
ax = fig.add_subplot(gs[0, 0])
ax.imshow(dog)
ax.set_title('224 x 224 input')
ax.axis('off')
grid = np.ones((14 * 18, 14 * 18, 3), np.uint8) * 255
for r in range(14):
    for c in range(14):
        grid[r*18:r*18+16, c*18:c*18+16] = patches[r, c]
ax = fig.add_subplot(gs[0, 1])
ax.imshow(grid)
ax.set_title('14 x 14 patches of 16 x 16 pixels')
ax.axis('off')
ax = fig.add_subplot(gs[1, :])
sequence = np.ones((16, 28 * 18, 3), np.uint8) * 255
for i in range(28):
    sequence[:, i*18:i*18+16] = patches[i // 14, i % 14]
ax.imshow(sequence)
ax.set_title('... a sequence of 196 tokens (the first two rows shown)')
ax.axis('off')
savefig('transformer_patches')
plt.show()

## Learned positional embeddings

The positional embeddings are learned from scratch, without any notion of 2D. Still, the embedding of each patch is most similar to those of patches in the same row and column, and nearby: the model has learned the 2D grid from the data (cf. Figure 7 of the ViT paper).

In [ ]:
pos = vit.encoder.pos_embedding[0, 1:].detach().cpu()          # (196, 768), without the class token
pos = F.normalize(pos, dim=1)
similarity = (pos @ pos.T).reshape(14, 14, 14, 14)

shown = [0, 3, 7, 10, 13]                                      # a subset of rows and columns
fig, axs = plt.subplots(len(shown), len(shown), figsize=(12, 12))
for i, r in enumerate(shown):
    for j, c in enumerate(shown):
        axs[i, j].imshow(similarity[r, c], cmap='viridis', vmin=-1, vmax=1)
        axs[i, j].set_xticks([])
        axs[i, j].set_yticks([])
    axs[i, 0].set_ylabel(f'row {r}')
for j, c in enumerate(shown):
    axs[-1, j].set_xlabel(f'column {c}')
fig.suptitle('Cosine similarity of the positional embedding of patch (row, column) to those of all 14x14 patches')
plt.subplots_adjust(wspace=0.05, hspace=0.05)
savefig('transformer_position_embeddings')
plt.show()

## How far do the heads look?

For every head we compute the *mean attention distance*: the distance (in pixels) between a query patch and the patches it attends to, weighted by the attention weights. It is the transformer's analogue of the receptive field of a CNN (Lecture 7). Unlike a CNN, a head can look at the whole image from the first layer on. In the early layers, some heads attend locally (a few patches) and others globally; in the last layers all heads are global.

In [ ]:
def attention_maps(image):
    x = preprocess(torch.from_numpy(image).permute(2, 0, 1))[None].to(device)
    maps = []
    with torch.no_grad():
        x = vit._process_input(x)
        x = torch.cat([vit.class_token.expand(1, -1, -1), x], dim=1) + vit.encoder.pos_embedding
        for layer in vit.encoder.layers:
            y = layer.ln_1(x)
            y, a = layer.self_attention(y, y, y, need_weights=True, average_attn_weights=False)
            maps.append(a[0, :, 1:, 1:].cpu())                       # (heads, 196, 196), patch-to-patch
            x = x + y
            x = x + layer.mlp(layer.ln_2(x))
    return maps

rows, cols = np.meshgrid(np.arange(14), np.arange(14), indexing='ij')
coords = np.stack([rows.ravel(), cols.ravel()], 1) * 16
distance = torch.from_numpy(np.linalg.norm(coords[:, None] - coords[None], axis=2)).float()   # (196, 196) pixels

images = [load(n) for n in ['image_dog.jpg', 'image_me.jpg', 'image_cc_1.jpg', 'image.jpg']]
mean_distance = np.zeros((12, 12))
for image in images:
    for l, a in enumerate(attention_maps(image)):
        a = a / a.sum(-1, keepdim=True)                              # renormalise without the class token
        mean_distance[l] += (a * distance).sum(-1).mean(-1).numpy() / len(images)

plt.figure(figsize=(8, 5))
for h in range(12):
    plt.scatter(np.arange(1, 13), mean_distance[:, h], s=20, c='tab:blue', alpha=0.7)
plt.xlabel('layer')
plt.ylabel('mean attention distance (pixels)')
plt.title('ViT-B/16: mean attention distance of each of the 12 heads')
plt.ylim(0, 224 * 0.6)
plt.grid(alpha=0.3)
savefig('transformer_attention_distance')
plt.show()

## Normalisation layers

Batch normalisation (CNNs) and layer normalisation (transformers) compute the same thing, $(x - \mu) / \sigma$ followed by a learned scale and shift, but over different axes: BatchNorm over the batch (and spatial positions) for each channel, LayerNorm over the channels of each token, independently of the rest of the batch.

In [ ]:
x = torch.randn(8, 197, 768) * 3 + 5          # (batch, tokens, channels)
ln = torch.nn.LayerNorm(768)(x)
bn = torch.nn.BatchNorm1d(768)(x.transpose(1, 2)).transpose(1, 2)   # BatchNorm1d expects (batch, channels, length)
print('LayerNorm: per token, mean/std over the channels ->', ln[0, 0].mean().item().__round__(3), ln[0, 0].std().item().__round__(3))
print('BatchNorm: per channel, mean/std over batch and tokens ->', bn[:, :, 0].mean().item().__round__(3), bn[:, :, 0].std().item().__round__(3))
print('LayerNorm output of one sample does not depend on the other samples:',
      torch.allclose(torch.nn.LayerNorm(768)(x[:1]), ln[:1], atol=1e-5))